In [ ]:
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeFace, BRepBuilderAPI_MakePolygon
from OCC.Core.BRepPrimAPI import BRepPrimAPI_MakeBox
from OCC.Core.gp import gp_Pnt
from OCC.Display.WebGl.jupyter_renderer import JupyterRenderer
from OCC.Core.BOPAlgo import BOPAlgo_Splitter

In [30]:
# See https://dev.opencascade.org/doc/overview/html/occt_user_guides__modeling_algos.html
def make_splitter_plane(pt1, pt2):
    x1, y1, z = pt1
    x2, y2, z = pt2
    W = BRepBuilderAPI_MakePolygon()
    W.Add(gp_Pnt(x1, y1, z))
    W.Add(gp_Pnt(x1, y2, z))
    W.Add(gp_Pnt(x2, y2, z))
    W.Add(gp_Pnt(x2, y1, z))
    W.Close()

    # At this point, the polygon shape is a "wire frame"
    shape1_wire = W.Shape()
    # print("Shape type of Polygon after .Close()", shape1_wire.ShapeType)
    # in order to perform a split with the enture surface of the polygon, we have to convert the wire frame to a face
    shape1_face = BRepBuilderAPI_MakeFace(shape1_wire)
    # print("Shape type of Face", shape1_face.Shape().ShapeType)

    return shape1_face

def split(component, tools):
    splitter = BOPAlgo_Splitter()
    splitter.SetNonDestructive(False)
    splitter.AddArgument(component)  # arugment means object to cut
    if isinstance(tools, BRepBuilderAPI_MakeFace):
        splitter.AddTool(tools.Shape())  # tool means arguments are cut by this
    else:
        for tool in tools:
            splitter.AddTool(tool.Shape())  # tool means arguments are cut by this
    splitter.Perform()

    result = splitter.Shape()
    return result

plane = make_splitter_plane((-2, -2, 1), (2, 2, 1))
box = BRepPrimAPI_MakeBox(gp_Pnt(-0.25, 0, 0), gp_Pnt(0.25, 1, 2)).Shape()
result = split(box, plane)

rnd = JupyterRenderer()
rnd.DisplayShape(result, render_edges=True)
rnd.Display()

In [31]:
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.IFSelect import IFSelect_RetDone

step_path = fr'C:\Users\Soske\Documents\git_projects\cavsim2d\rect_split_test.stp'
reader = STEPControl_Reader()
status = reader.ReadFile(step_path)
reader.SetSystemLengthUnit(1000) # units to mm

if status != IFSelect_RetDone:
    raise RuntimeError("STEP file could not be read.")

# Transfer the roots to obtain a single compound shape
reader.TransferRoots()
rectwg = reader.OneShape()

# mid_plane = make_splitter_plane((-0.075, -0.050, 0), (0.075, 0.050, 0))
plane1 = make_splitter_plane((-0.075, -0.050, -0.1), (0.075, 0.050, -0.1))
plane2 = make_splitter_plane((-0.075, -0.050, 0.1), (0.075, 0.050, 0.1))

print(type(plane1))
print(type(plane2))
result = split(rectwg, [plane1, plane2])

rnd = JupyterRenderer()
# rnd.DisplayShape(plane.Shape(), render_edges=True)
# rnd.DisplayShape(rectwg, render_edges=True)
rnd.DisplayShape(result, render_edges=True)
rnd.Display()

<class 'OCC.Core.BRepBuilderAPI.BRepBuilderAPI_MakeFace'>
<class 'OCC.Core.BRepBuilderAPI.BRepBuilderAPI_MakeFace'>


In [14]:
# from OCC.Core.STEPControl import STEPControl_Reader
# from OCC.Core.IFSelect import IFSelect_RetDone
# from OCC.Display.WebGl.jupyter_renderer import JupyterRenderer
#
# # step_path = fr'C:\Users\Soske\Documents\CST\Youtube\tesla_from_cst.stp'
# # step_path = fr'C:\Users\Soske\Documents\git_projects\cavsim2d\pillbox_from_cst.stp'
# # step_path = fr'C:\Users\Soske\Documents\Rostock Files\phd models\c3794_4hc_1fpc.stp'
# # step_path = fr'C:\Users\Soske\Documents\Rostock Files\phd models\c3794.stp'
# step_path = fr'C:\Users\Soske\Documents\git_projects\cavsim2d\rect_split_test.stp'
# reader = STEPControl_Reader()
# status = reader.ReadFile(step_path)
#
# if status != IFSelect_RetDone:
#     raise RuntimeError("STEP file could not be read.")
#
# # Transfer the roots to obtain a single compound shape
# reader.TransferRoots()
# shape = reader.OneShape()
#
# rnd = JupyterRenderer()
# rnd.DisplayShape(shape, render_edges=True)
# rnd.Display()


In [32]:
from OCC.Core.STEPControl import STEPControl_Writer, STEPControl_AsIs
from OCC.Core.Interface import Interface_Static

# optional: write in AP203 (old) or AP214
Interface_Static.SetCVal("write.step.schema", "AP214")

shape = result    # your OCC result

writer = STEPControl_Writer()
writer.Transfer(shape, STEPControl_AsIs)
writer.Write("my_geometry.step")

1

In [35]:
from netgen.occ import *
from netgen.meshing import MeshingParameters
from ngsolve import *
from netgen.webgui import *

# Wrap the OpenCascade shape
# geom = OCCGeometry(result)
geom = OCCGeometry(fr"C:\Users\Soske\Documents\git_projects\cavsim2d\my_geometry.step")
geom = Glue([solid for solid in geom.solids])

nsolids = len(geom.solids)
for i, solid in enumerate(geom.solids):
    matname = f"cell_{i+1}"  # or any name you want
    geom.solids[i].mat(matname)
    geom.solids[i].faces.col = (i%nsolids, i%(nsolids-1), 1)
    print(solid.bounding_box)
    geom.solids[i].faces.Max(Z).name = f'port{i+1}'
    if i == nsolids-1:
        geom.solids[i].faces.Min(Z).name = fr'port{i+2}'

# Draw(geom)

mp = MeshingParameters(maxh=100)
ngmesh = OCCGeometry(geom).GenerateMesh(mp)

# Convert to NGSolve mesh
mesh = Mesh(ngmesh)
print(mesh.GetMaterials())
print(mesh.GetBoundaries())
Draw(ngmesh, materials={"default": "orange", "niobium": "blue"})

((-0.0500001, -0.0250001, 0.0999999), (0.0500001, 0.0250001, 0.2))
((-0.0500001, -0.0250001, -0.1), (0.0500001, 0.0250001, 0.1))
((-0.0500001, -0.0250001, -0.2), (0.0500001, 0.0250001, -0.0999999))
('cell_1', 'cell_2', 'cell_3')
('port1', 'default', 'default', 'default', 'default', 'port2', 'default', 'default', 'port3', 'default', 'default', 'default', 'default', 'port4', 'default', 'default')


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'mesh_dim': 3, 'mesh_cente…

BaseWebGuiScene

In [7]:
# from ngsolve import *
# from netgen.occ import *
# from ngsolve.webgui import Draw
# import numpy as np
#
# port = 'port2'
# fesport = HCurl(mesh, order=3, dirichlet='default', definedon=mesh.Boundaries(port))
# fesfull = HCurl(mesh, order=3, dirichlet='default')
#
# u, v = fesport.TnT()
# a = BilinearForm(curl(u.Trace())*curl(v.Trace())*ds(port))
# m = BilinearForm(u.Trace()*v.Trace()*ds(port))
# apre = BilinearForm((curl(u).Trace()*curl(v).Trace() + u.Trace()*v.Trace())*ds(port))
# pre = Preconditioner(apre, type="direct", inverse="sparsecholesky")
#
# a.Assemble()
# m.Assemble()
# apre.Assemble()
#
# settings = {'Objects': {'Clipping Plane': False, 'Vectors': True}, 'Colormap': {'ncolors': 125},
#             'Clipping': {'enable': True, 'x': 1, 'y': 0, 'z': 0},
#            'Vectors': {'grid_size': 200}}
# G, fesh1 = fesport.CreateGradient()
# GT = G.CreateTranspose()
# math1 = GT @ m.mat @ G
# invh1 = math1.Inverse(inverse="sparsecholesky", freedofs=fesh1.FreeDofs())
#
# proj = IdentityMatrix(fesport.ndof) - G@invh1@GT@m.mat
#
# projpre = proj@pre
# evals, evecs = solvers.PINVIT(a.mat, m.mat, pre=projpre, num=5, maxit=30,
#                               printrates=False)
# print(evals)
#
# efield = GridFunction(fesport)
# efield.vec.data = evecs[0]
# Draw(Norm(efield), mesh, settings=settings)
# ex, ey = efield[0], efield[1]


In [38]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

settings = {'Objects': {'Clipping Plane': True, 'Vectors': True}, 'Colormap': {'ncolors': 250},
            'Clipping': {'enable': True, 'x': 1, 'y': 0, 'z': 0},
           'Vectors': {'grid_size': 200}}

cell = 'cell_3'
fesfull = HCurl(mesh, order=3, dirichlet='default', definedon=mesh.Materials(cell))

u, v = fesfull.TnT()
a = BilinearForm(curl(u)*curl(v)*dx(cell))
m = BilinearForm(u*v*dx(cell))
apre = BilinearForm((curl(u)*curl(v) + u*v)*dx(cell))
pre = Preconditioner(apre, type="direct", inverse="sparsecholesky")

a.Assemble()
m.Assemble()
apre.Assemble()

G, fesh1 = fesfull.CreateGradient()
GT = G.CreateTranspose()
math1 = GT @ m.mat @ G
invh1 = math1.Inverse(inverse="sparsecholesky", freedofs=fesh1.FreeDofs())

proj = IdentityMatrix(fesfull.ndof) - G@invh1@GT@m.mat

projpre = proj@pre
evals, evecs = solvers.PINVIT(a.mat, m.mat, pre=projpre, num=5, maxit=30,
                              printrates=False)
print(evals)

efield = GridFunction(fesfull)
efield.vec.data = evecs[0]
Draw(Norm(BoundaryFromVolumeCF(efield)), mesh, settings=settings)

ex, ey = efield[0], efield[1]


 988.215
 1974.71
 3972.61
 4008.18
 4987.44



WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Objects': {'Clipping Plane':…